# CollegeWise: Exploratory Data Analysis (EDA)

This notebook conducts in-depth exploratory analysis on the curated dataset of Indian colleges (~2,500 institutions across 35 States & Union Territories) combining official records from **AICTE** and the **National Institutional Ranking Framework (NIRF)**.

### Core Analytical Focus:
1. Geographic & state-level representation
2. Ownership models & institutional classification
3. Empirical distributions of placement rate & median compensation
4. Fee structures & economic affordability
5. Missing value audit and transparency report
6. Cross-dimensional correlations among engineered metrics

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.dpi'] = 120

data_path = '../data/processed/colleges_features.parquet'
df = pd.read_parquet(data_path)
print(f"Loaded dataset with {len(df)} rows and {len(df.columns)} columns.")
df.head(3)

## 1. Geographic & State-Wise Distribution

In [ ]:
plt.figure(figsize=(10, 6))
top_states = df['state'].value_counts().head(15)
sns.barplot(x=top_states.values, y=top_states.index, hue=top_states.index, palette='mako', legend=False)
plt.title('Top 15 States by Institutional Representation')
plt.xlabel('Count of Colleges')
plt.ylabel('State')
plt.show()

## 2. Institutional Ownership & Categories

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
own = df['ownership'].value_counts()
ax1.pie(own.values, labels=own.index, autopct='%1.1f%%', colors=['#A5D6A7', '#CE93D8'], startangle=140)
ax1.set_title('Institutional Ownership Breakdown')

inst_types = df['institution_type'].value_counts().head(5)
sns.barplot(x=inst_types.values, y=inst_types.index, ax=ax2, hue=inst_types.index, palette='crest', legend=False)
ax2.set_title('Top Institution Types')
plt.tight_layout()
plt.show()

## 3. Placement & Median Salary Distributions

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
pl_data = df['placement_rate'].dropna()
sns.histplot(pl_data, kde=True, bins=20, color='#26A69A', ax=ax1)
ax1.set_title('Placement Rate (%) Distribution')
ax1.set_xlabel('Placement Rate (%)')

sal_data = df['median_package_lpa'].dropna()
sns.histplot(sal_data, kde=True, bins=20, color='#AB47BC', ax=ax2)
ax2.set_title('Median Package (LPA) Distribution')
ax2.set_xlabel('Median Package (Lakhs Per Annum)')
plt.tight_layout()
plt.show()

## 4. Tuition Fee Analysis & ROI Landscape

In [ ]:
plt.figure(figsize=(9, 5))
roi_df = df.dropna(subset=['tuition_fee_annual', 'placement_rate'])
sns.scatterplot(data=roi_df, x='tuition_fee_annual', y='placement_rate', hue='ownership', alpha=0.8)
plt.title('Placement Rate vs. Annual Tuition Fee (ROI Landscape)')
plt.xlabel('Annual Tuition Fee (INR)')
plt.ylabel('Placement Rate (%)')
plt.show()

## 5. Correlation Analysis of Normalized Preference Dimensions

In [ ]:
score_cols = ['placement_score', 'academic_score', 'affordability_score', 'infrastructure_score', 'location_score', 'student_life_score']
corr = df[score_cols].corr()
plt.figure(figsize=(7, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='vlag', vmin=-1, vmax=1)
plt.title('Correlation Matrix: 6 Core Decision Dimensions')
plt.show()

## 6. Transparency & Missing Value Audit

In [ ]:
missing = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False).head(10)
print('--- Top Missing Fields (%) ---')
print(missing.to_string())
